In [ ]:
import sys

sys.path.append("../pgm")
from probabilistic_graph_models import ProbGraphModel
from influence_diagrams import IDGUM

In [ ]:
import pyagrum as gum
import pyagrum.lib.notebook as gnb
from IPython.display import display, HTML

In [ ]:
import math
from itertools import chain

# a function to show results on decision nodes T and D
def show_decisions(ie):
    diag = ie.influenceDiagram()
    decisions = [diag.variable(item).name() for item in diag.getDecisionGraph()]
    optimal_decisions = [ie.optimalDecision(item) for item in decisions]
    posterior = [ie.posterior(item) for item in decisions]
    posterior_utility = [ie.posteriorUtility(item) for item in decisions]
    post_list = chain.from_iterable(zip(posterior, posterior_utility))

    mean = ie.MEU()['mean']
    std = math.sqrt(ie.MEU()['variance'])
    risk = std / mean
    
    caption_1 = [f"Strategy for {item}" for item in decisions] + \
        ["MEU, its standard deviation, and associated risk"]
    caption_2 = list(chain.from_iterable(
        zip([f"Final decision for {item}" for item in decisions], \
            [f"Final reward for {item}" for item in decisions])
    ))


    display(HTML("<h2>Inference in the LIMID optimizing the decisions nodes</h2>"))
    gnb.flow.row(
        *optimal_decisions, 
        f"mean: {mean:5.3f} -- std: {std:5.3f} -- risk: {risk:5.3f}",
        captions=caption_1,
                )

    gnb.flow.row(
        *post_list,
        captions=caption_2,
    )

In [ ]:
model = ProbGraphModel.read_model("models/id_oil_wildcatter.yaml")

In [ ]:
gum_model = IDGUM(network=model)

In [ ]:
print(model.name)
model.print_variables()
model.print_nodes()
model.print_arcs()
model.print_potentials()
print(model.get_variable_by_name("Purchase"))
print(model.get_potential_by_name("Base profit"))

In [ ]:
gum_model.model

In [ ]:
gum_model.model.names()

In [ ]:
gum_model.model.idFromName("Reward")

In [ ]:
gum_model.model.ids(list(gum_model.model.names()))

In [ ]:
gum_model.model.variableFromName("Reward").__dict__

In [ ]:
gum_model.model.nodes()

In [ ]:
gum_model.model.idFromName("Drilling")

In [ ]:
gum_model.model.variableFromName("Drilling").description()

In [ ]:
gum_model.model.variableFromName("Drilling").domain()

In [ ]:
gum_model.model.arcs()

In [ ]:
gum_model.model.getDecisionGraph()

In [ ]:
help(gum_model.model.ids)

In [ ]:
gnb.flow.row(gum_model.model, gnb.getInference(gum_model.model))

In [ ]:
gum_model.model.saveBIFXML("bifxml/oil_wildcatter_test.BIFXML")

In [ ]:
oil = gum.loadID("bifxml/oil_wildcatter.BIFXML")

In [ ]:
gnb.flow.row(oil, gnb.getInference(oil))

In [ ]:
gnb.flow.row(oil.cpt("OilContents"), gum_model.model.cpt("Oil content"))

In [ ]:
gnb.flow.row(oil.cpt("TestResult"), gum_model.model.cpt("Test result"))

In [ ]:
decision_graph = gum_model.model.getDecisionGraph()
decision_names = [gum_model.model.variable(item).name() for item in decision_graph]
print(decision_names)
for decision in decision_graph:
    print(decision, gum_model.model.variable(decision).name())

In [ ]:
ie = gum.ShaferShenoyLIMIDInference(oil)
ie.makeInference()
show_decisions(ie)

In [ ]:
ie = gum.ShaferShenoyLIMIDInference(gum_model.model)
ie.makeInference()
show_decisions(ie)